# Unified No-Scaler Clustering and Hierarchical Sensitivity Analysis

This notebook reproduces the revised analysis requested during peer review.

## Primary computational workflow

```text
Archived frozen feature table
→ no StandardScaler
→ PCA 50 dimensions using full SVD
→ KMeans in PCA space
```

KMeans settings: `init="k-means++"`, `n_init=20`, `max_iter=500`, `tol=1e-4`, and `random_state=42`.

The term **post hoc** refers only to assigning an architectural description to an entire computational cluster after representative-sample inspection. Individual components are not manually transferred between clusters.

## Statistical-unit correction

- Facade descriptors are averaged within each building to obtain 59 independent building records.
- Component morphology compositions are calculated per building.
- Component-level contingency tables are retained as descriptive summaries only.
- Building-level composition inference uses Hellinger-transformed proportions and permutation testing.


In [ ]:
from pathlib import Path
from itertools import permutations
import re

import numpy as np
import pandas as pd

from scipy.stats import rankdata, chi2 as chi2_distribution
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
PCA_COMPONENTS = 50
K_VALUES = range(2, 9)
FINAL_K = {"Window": 3, "Door": 2, "Column": 2}
COMPONENT_ORDER = ["Window", "Door", "Column"]
CONCESSION_ORDER = ["British", "French", "German", "Russian", "Japanese"]

CLUSTER_NAMES = {
    "Window": {1: "Arched/ornamented tendency", 2: "Horizontal-organization tendency", 3: "Rectilinear tendency"},
    "Door": {1: "Shutter/grille-feature grouping", 2: "Conventional door-leaf grouping"},
    "Column": {1: "Brick-textured/articulated grouping", 2: "Plain-rectilinear grouping"},
}

CONCESSION_CN_EN = {
    "英租界": "British", "法租界": "French", "德租界": "German",
    "俄租界": "Russian", "日租界": "Japanese",
}


In [ ]:
BASE_DIR = Path.cwd()

RESNET_CSV = BASE_DIR / "data" / "Component_ResNet50_Features_2456x2048.csv"
DINO_CSV = BASE_DIR / "data" / "Component_DINOv2_Features_2456x384.csv"
MAPPING_XLSX = BASE_DIR / "data" / "Building_Facade_Mapping_Final.xlsx"
OPENCV_XLSX = BASE_DIR / "data" / "OpenCV_Facade_Features_Final_Corrected.xlsx"

OUTPUT_DIR = BASE_DIR / "results" / "Unified_NoScaler_Reviewer_Rerun_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [RESNET_CSV, DINO_CSV, MAPPING_XLSX, OPENCV_XLSX]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output:", OUTPUT_DIR.resolve())


## 1. Load the archived feature tables and the final manually verified mapping

In [ ]:
resnet = pd.read_csv(RESNET_CSV)
dino = pd.read_csv(DINO_CSV)
mapping = pd.read_excel(MAPPING_XLSX, sheet_name="Building_Facade_Mapping")
building_map = pd.read_excel(MAPPING_XLSX, sheet_name="建筑_立面数量统计")
opencv = pd.read_excel(OPENCV_XLSX)

resnet_features = sorted(
    [c for c in resnet.columns if re.fullmatch(r"feature_\d+", c)],
    key=lambda c: int(c.split("_")[1]),
)
dino_features = sorted(
    [c for c in dino.columns if re.fullmatch(r"dinov2_feature_\d+", c)],
    key=lambda c: int(c.split("_")[-1]),
)

assert len(resnet) == 2456 and len(resnet_features) == 2048
assert len(dino) == 2456 and len(dino_features) == 384
assert mapping["facade_id"].is_unique

facade_map = mapping[[
    "facade_id", "building_id", "master_building_name", "master_concession",
    "master_address", "master_year"
]].rename(columns={
    "building_id": "master_building_id",
    "master_building_name": "building_name_master",
    "master_concession": "concession_master",
    "master_address": "address_master",
    "master_year": "year_master",
})


## 2. Rebuild the primary ResNet50 clustering without StandardScaler

In [ ]:
def run_feature_clustering(data, feature_columns, model_name):
    prepared = {}
    validation_rows = []
    size_rows = []
    final_rows = []

    for component_type in COMPONENT_ORDER:
        sub = data[data["component_type"] == component_type].copy().reset_index(drop=True)
        X = sub[feature_columns].to_numpy(dtype=np.float32)

        pca = PCA(
            n_components=min(PCA_COMPONENTS, len(sub) - 1, X.shape[1]),
            svd_solver="full",
        )
        X_pca = pca.fit_transform(X).astype(np.float32)
        prepared[component_type] = {"data": sub, "X_pca": X_pca, "pca": pca}

        for k in K_VALUES:
            model = KMeans(
                n_clusters=k,
                init="k-means++",
                n_init=20,
                max_iter=500,
                tol=1e-4,
                random_state=RANDOM_STATE,
            )
            labels = model.fit_predict(X_pca)
            counts = pd.Series(labels + 1).value_counts().sort_index()

            validation_rows.append({
                "model": model_name,
                "component_type": component_type,
                "K": k,
                "Silhouette": silhouette_score(X_pca, labels),
                "Calinski_Harabasz": calinski_harabasz_score(X_pca, labels),
                "Davies_Bouldin": davies_bouldin_score(X_pca, labels),
                "Inertia": model.inertia_,
                "PCA_explained_variance": pca.explained_variance_ratio_.sum(),
            })

            for cluster_id, n in counts.items():
                size_rows.append({
                    "model": model_name,
                    "component_type": component_type,
                    "K": k,
                    "Cluster": int(cluster_id),
                    "n": int(n),
                    "proportion": n / len(sub),
                })

        selected_k = FINAL_K[component_type]
        selected = KMeans(
            n_clusters=selected_k,
            init="k-means++",
            n_init=20,
            max_iter=500,
            tol=1e-4,
            random_state=RANDOM_STATE,
        )
        labels = selected.fit_predict(X_pca) + 1

        result = sub[[
            "component_id", "building_id", "facade_id", "building_name_cn",
            "component_type", "image_name"
        ]].copy()
        result[f"{model_name}_Cluster"] = labels
        result[f"{model_name}_Distance_to_Center"] = np.linalg.norm(
            X_pca - selected.cluster_centers_[labels - 1], axis=1
        )
        if model_name == "ResNet50":
            result["Cluster_Informed_Visual_Grouping"] = result[
                f"{model_name}_Cluster"
            ].map(CLUSTER_NAMES[component_type])
        final_rows.append(result)

    return (
        prepared,
        pd.DataFrame(validation_rows),
        pd.DataFrame(size_rows),
        pd.concat(final_rows, ignore_index=True),
    )

resnet_prepared, resnet_validation, resnet_sizes, resnet_assignments = run_feature_clustering(
    resnet, resnet_features, "ResNet50"
)

resnet_assignments = resnet_assignments.merge(
    facade_map, on="facade_id", how="left", validate="many_to_one"
)
resnet_assignments["concession"] = resnet_assignments["concession_master"].map(CONCESSION_CN_EN)

print(resnet_assignments.groupby([
    "component_type", "ResNet50_Cluster", "Cluster_Informed_Visual_Grouping"
]).size())


## 3. Corrected component-level descriptive contingency tables

In [ ]:
def adjusted_standardized_residuals(observed, expected):
    observed = np.asarray(observed, dtype=float)
    expected = np.asarray(expected, dtype=float)
    n = observed.sum()
    row_prop = observed.sum(axis=1, keepdims=True) / n
    col_prop = observed.sum(axis=0, keepdims=True) / n
    return (observed - expected) / np.sqrt(expected * (1 - row_prop) * (1 - col_prop))

component_tables = {}
for component_type in COMPONENT_ORDER:
    names = list(CLUSTER_NAMES[component_type].values())
    sub = resnet_assignments[resnet_assignments["component_type"] == component_type]
    table = pd.crosstab(
        sub["concession"], sub["Cluster_Informed_Visual_Grouping"]
    ).reindex(index=CONCESSION_ORDER, columns=names, fill_value=0)
    component_tables[component_type] = table
    print("\n", component_type)
    display(table)

# Retain these tables as descriptive because components are nested in buildings.


## 4. Building-level component composition and permutation sensitivity

In [ ]:
building_base = building_map[[
    "building_id", "master_building_name", "master_concession", "facade_count"
]].rename(columns={
    "master_building_name": "building_name",
    "master_concession": "concession_cn",
})
building_base["concession"] = building_base["concession_cn"].map(CONCESSION_CN_EN)

building_tables = {}
for component_type in COMPONENT_ORDER:
    names = list(CLUSTER_NAMES[component_type].values())
    sub = resnet_assignments[resnet_assignments["component_type"] == component_type]
    counts = pd.crosstab(
        sub["master_building_id"], sub["Cluster_Informed_Visual_Grouping"]
    ).reindex(index=building_base["building_id"], columns=names, fill_value=0)
    total = counts.sum(axis=1)
    proportions = counts.div(total.replace(0, np.nan), axis=0)
    table = building_base.set_index("building_id").join(
        counts.add_prefix("n__")
    ).join(proportions.add_prefix("prop__"))
    table[f"{component_type}_total_n"] = total
    building_tables[component_type] = table.reset_index()


In [ ]:
def pseudo_f_euclidean(X, labels):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    grand_mean = X.mean(axis=0)
    ss_total = ((X - grand_mean) ** 2).sum()
    groups = np.unique(labels)
    ss_between = sum(
        len(X[labels == g]) * ((X[labels == g].mean(axis=0) - grand_mean) ** 2).sum()
        for g in groups
    )
    ss_within = ss_total - ss_between
    pseudo_f = (ss_between / (len(groups) - 1)) / (ss_within / (len(X) - len(groups)))
    return pseudo_f, ss_between / ss_total


def permutation_permanova(X, labels, n_permutations=9999, seed=42):
    observed_f, r2 = pseudo_f_euclidean(X, labels)
    rng = np.random.default_rng(seed)
    null = np.array([
        pseudo_f_euclidean(X, rng.permutation(labels))[0]
        for _ in range(n_permutations)
    ])
    p_value = (np.sum(null >= observed_f) + 1) / (n_permutations + 1)
    return observed_f, r2, p_value

permanova_rows = []
for component_type in COMPONENT_ORDER:
    table = building_tables[component_type]
    names = list(CLUSTER_NAMES[component_type].values())
    prop_cols = [f"prop__{name}" for name in names]
    valid = table[table[f"{component_type}_total_n"] > 0].copy()
    X = np.sqrt(valid[prop_cols].to_numpy(dtype=float))
    labels = valid["concession"].to_numpy()
    F, R2, p = permutation_permanova(X, labels, 9999, RANDOM_STATE)
    permanova_rows.append({
        "component_type": component_type,
        "N_buildings": len(valid),
        "pseudo_F": F,
        "R2": R2,
        "permutation_p": p,
    })

building_permanova = pd.DataFrame(permanova_rows)
display(building_permanova)


## 5. Aggregate the 77 facade records to 59 independent buildings

In [ ]:
INDICATORS = [
    "窗户数量", "单窗平均面积", "窗宽高比均值", "窗户总面积占比",
    "门数量", "单门平均面积", "门宽高比均值", "门总面积占比",
    "立柱数量", "单柱平均面积", "柱宽厚比均值", "立柱总面积占比",
    "img_wh_ratio", "edge_pixel_ratio", "window_area_ratio", "contour_count",
    "curve_contour_ratio", "symmetry_degree",
]

opencv_corrected = opencv.merge(
    facade_map, on="facade_id", how="left", validate="one_to_one"
)
opencv_corrected["concession"] = opencv_corrected["concession_master"].map(CONCESSION_CN_EN)

building_opencv = opencv_corrected.groupby([
    "master_building_id", "building_name_master", "concession"
], as_index=False)[INDICATORS].mean()

assert len(building_opencv) == 59
building_opencv.to_csv(OUTPUT_DIR / "Building_Level_OpenCV_59.csv", index=False, encoding="utf-8-sig")


## 6. Unified DINOv2 comparison under the same no-scaler preprocessing

In [ ]:
dino_prepared, dino_validation, dino_sizes, dino_assignments = run_feature_clustering(
    dino, dino_features, "DINOv2"
)


def best_mapping(reference, target):
    reference_values = sorted(np.unique(reference))
    target_values = sorted(np.unique(target))
    best = None
    best_score = -1
    for perm in permutations(reference_values):
        mapping = dict(zip(target_values, perm))
        aligned = np.array([mapping[x] for x in target])
        score = np.sum(aligned == reference)
        if score > best_score:
            best = mapping
            best_score = score
    return best

agreement_rows = []
for component_type in COMPONENT_ORDER:
    r = resnet_assignments[
        resnet_assignments["component_type"] == component_type
    ][["component_id", "ResNet50_Cluster"]]
    d = dino_assignments[
        dino_assignments["component_type"] == component_type
    ][["component_id", "DINOv2_Cluster"]]
    merged = r.merge(d, on="component_id", validate="one_to_one")
    y_r = merged["ResNet50_Cluster"].to_numpy()
    y_d = merged["DINOv2_Cluster"].to_numpy()
    mapping = best_mapping(y_r, y_d)
    aligned = np.array([mapping[x] for x in y_d])
    agreement_rows.append({
        "component_type": component_type,
        "ARI": adjusted_rand_score(y_r, y_d),
        "NMI": normalized_mutual_info_score(y_r, y_d),
        "aligned_agreement": np.mean(y_r == aligned),
    })

agreement = pd.DataFrame(agreement_rows)
display(agreement)


## 7. Export core outputs

In [ ]:
resnet_validation.to_csv(OUTPUT_DIR / "ResNet50_K2_K8_Validation.csv", index=False, encoding="utf-8-sig")
resnet_assignments.to_csv(OUTPUT_DIR / "ResNet50_Final_Assignments_Corrected.csv", index=False, encoding="utf-8-sig")
building_permanova.to_csv(OUTPUT_DIR / "Building_Level_Component_PERMANOVA.csv", index=False, encoding="utf-8-sig")
dino_validation.to_csv(OUTPUT_DIR / "DINOv2_K2_K8_Validation_NoScaler.csv", index=False, encoding="utf-8-sig")
agreement.to_csv(OUTPUT_DIR / "ResNet50_DINOv2_Agreement_NoScaler.csv", index=False, encoding="utf-8-sig")

print("Core outputs saved.")
